## OLLAMA LIBRARY 

In [ ]:
pip install ollama 

In [ ]:
import ollama

#### GENERATE FUNCTION

In [ ]:
# GENERATION
response = ollama.generate(model="llama3.2:1b",prompt="why does moon glow ?")
response

In [ ]:
print(response.response)

In [ ]:
# STREAM RESPONSE
response = ollama.generate(model="llama3.2:1b",prompt="why does moon glow ?", stream=True)
for i in response:
    print(i["response"], end="")

In [ ]:
# GIVING MULTIMODEL INPUT TO MODEL [PASSING IMAGE TO THE MODEL]

import base64
import ollama

image_path= "Linkedin.jpg"

with open(image_path,"rb") as f: # OPENING THE IMAGE IN READ BINARY 
    image_bytes= f.read()    # READING THE IMAGE 
image_64= base64.b64encode(image_bytes).decode("utf-8") # ENCODING THE IMAGE IN BASE 64 BITS.

response= ollama.generate(model="gemma3:4b", images=[image_64], prompt="Give caption for the image.")
print(response.response)

In [ ]:
# PASSING MULTIPLE IMAGES TO THE MODEL.

image_paths = ["Linkedin.jpg","Green AI.png"]

images_base64=[]
for i in image_paths:
    with open(i , "rb") as f:
        image_bytes= f.read()
        images_base64.append(base64.b64encode(image_bytes).decode("utf-8"))

response= ollama.generate(model="gemma3:4b", images=images_base64,
                          prompt="Generate an story based on these images, make sure you take context from each and every image.")
print(response.response)

In [ ]:
# SYSTEM INSTRUCTIONS
response = ollama.generate(model="llama3.2:1b",prompt="why does moon glow ?", system="You are an funny assistant , you explain things in funny way")
print(response.response)

In [ ]:
response

In [ ]:
# OPTIONS PARAMETER
response = ollama.generate(model="llama3.2:1b",prompt="why is the ocean blue",
                           options={
                               "temperature":0.3,
                               "top_p":0.5,
                               "top_k":45
                           })


print(response.response)

In [ ]:
#OLLAMA LIST
local_models= ollama.list()
print(local_models)

print("**************************")

for i in local_models["models"]:
    print(i["model"])
    print(i["size"])


In [ ]:
#OLLAMA PULL
# model_name="deepseek-r1"
# progess = ollama.pull(model_name, stream=True)

# for i in progess:
#     print(i)


In [ ]:
#OLLAMA SHOW
models_details = ollama.show("gemma3:4b") # qwen3:8b
#print(models_details)
print(models_details.dict())

In [ ]:
model_dict= models_details.dict()
print(model_dict["capabilities"])
print(model_dict["parameters"])

## TOOL CALLING

In [ ]:
import ollama 

# TOOL 1: Mock Database for Inventory
def check_inventory(product_name: str):
    """Checks the local database for stock levels of a specific product."""
    inventory_db = {
        "laptop": {"stock": 5, "base_price": 1200},
        "monitor": {"stock": 0, "base_price": 300},
        "keyboard": {"stock": 25, "base_price": 80}
    }
    product = product_name.lower()
    if product in inventory_db:
        return inventory_db[product]
    return {"error": "Product not found"}




# TOOL 2: Business Logic for Discounts
def calculate_loyalty_discount(base_price: float, years_as_customer: int):
    """Calculates a custom discount: 5% per year, capped at 25%."""
    discount_pct = min(years_as_customer * 0.05, 0.25)
    final_price = base_price * (1 - discount_pct)
    return {"final_price": final_price, "discount_applied": f"{discount_pct*100}%"}

In [ ]:
# 1. Map the functions so the script can call them by name
available_functions = {
    'check_inventory': check_inventory,
    'calculate_loyalty_discount': calculate_loyalty_discount,
}

In [ ]:
# 1. Define the tool schemas for the LLM
tools = [
    {
        'type': 'function',
        'function': {
            'name': 'check_inventory',
            'description': 'Get stock and price for a product',
            'parameters': {
                'type': 'object',
                'properties': {
                    'product_name': {'type': 'string', 'description': 'Name of the product'}
                },
                'required': ['product_name'],
            },
        },
    },
    {
        'type': 'function',
        'function': {
            'name': 'calculate_loyalty_discount',
            'description': 'Calculate final price based on customer loyalty years',
            'parameters': {
                'type': 'object',
                'properties': {
                    'base_price': {'type': 'number', 'description': 'The original price'},
                    'years_as_customer': {'type': 'integer', 'description': 'How many years they have been a member'}
                },
                'required': ['base_price', 'years_as_customer'],
            },
        },
    }
]


In [ ]:
tools

In [ ]:
# 3. Execution Loop
messages = [{'role': 'user', 'content': 'Do we have laptops in stock? If so, what is the price for a customer who has been with us for 3 years?'}]

# First call: LLM decides which tools to use
response = ollama.chat(model="granite4.1:3b", 
        # qwen3:8b, 
        messages=messages, tools=tools)

In [ ]:
response

In [ ]:
response["message"]

In [ ]:
# Process tool calls
if response['message'].get('tool_calls'):
    for tool in response['message']['tool_calls']:
        function_to_call = available_functions[tool['function']['name']]
        args = tool['function']['arguments']
        
        # Execute local Python code
        result = function_to_call(**args)
        
        # Add the tool's output back to the conversation
        messages.append(response['message'])
        messages.append({'role': 'tool', 'content': str(result)})

messages

In [ ]:
# Final call: LLM summarizes the findings
final_response = ollama.chat(model="granite4.1:3b", 
        # qwen3:8b, 
        messages=messages)
print(final_response['message']['content'])

In [ ]:
## MODEL FILE